# Models - KMeans Clustering


### import DATA

In [22]:
%run "EDA_DATA.ipynb"

# Sau dòng trên, rfm_df (được tạo trong eda.ipynb) sẽ có sẵn trong notebook này.
assert 'rfm_df' in dir(), "Không tìm thấy rfm_df sau khi chạy eda.ipynb — kiểm tra lại tên biến/đường dẫn file."
print("Đã nạp rfm_df từ eda.ipynb, shape:", rfm_df.shape)


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\nghah\\OneDrive\\Documents\\dataset\\olist_orders_dataset.csv'

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\nghah\\OneDrive\\Documents\\dataset\\olist_orders_dataset.csv'

In [ ]:
import numpy as np

rfm_model = rfm_df.copy()

# Log-transform để giảm lệch cho Frequency & Monetary (thêm 1 để tránh log(0))
# LƯU Ý: Recency KHÔNG log-transform vì skewness ban đầu đã thấp (~0.45, gần chuẩn),
# nên giữ nguyên recency_days. Đặt lại tên biến bên dưới để không gây hiểu nhầm là đã log-transform.
rfm_model['recency_log'] = rfm_model['recency_days']  # thực chất KHÔNG log, giữ nguyên recency_days (xem lưu ý trên)
rfm_model['frequency_log'] = np.log1p(rfm_model['purchase_frequency'])
rfm_model['monetary_log']  = np.log1p(rfm_model['monetary_total'])


In [ ]:
rfm_model = rfm_df.copy()

# Skewness trước Log Transform
skew_before = rfm_model[['recency_days', 'purchase_frequency', 'monetary_total']].skew()

# Log Transform
rfm_model['recency_log'] = rfm_model['recency_days']
rfm_model['frequency_log'] = np.log1p(rfm_model['purchase_frequency'])
rfm_model['monetary_log'] = np.log1p(rfm_model['monetary_total'])

# Skewness sau Log Transform
skew_after = rfm_model[['recency_log', 'frequency_log', 'monetary_log']].skew()

# So sánh
comparison = pd.DataFrame({
    'Before Log': skew_before.values,
    'After Log': skew_after.values
}, index=['Recency', 'Frequency', 'Monetary'])

print(comparison.round(3))

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['recency_days','frequency_log', 'monetary_log']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_model[features])

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

# ⚙️ Bật/tắt bước dò K: để False sẽ BỎ QUA toàn bộ vòng lặp tốn thời gian bên dưới
# và dùng thẳng K=4 (đã được xác nhận trước đó). Chỉ bật True khi cần kiểm chứng lại
# (ví dụ đổi bộ feature, đổi dữ liệu, hoặc muốn có biểu đồ Elbow/Silhouette/DBI cho báo cáo).
RUN_K_SELECTION = False
DEFAULT_K = 4

if RUN_K_SELECTION:
    K_range = range(2, 11)
    N_INIT_SEARCH = 5          # giảm từ 10 -> 5 cho bước dò K (model cuối K=4 vẫn dùng n_init=10)
    SILHOUETTE_SAMPLE_SIZE = 5000  # tính silhouette trên mẫu ngẫu nhiên thay vì toàn bộ dữ liệu (None = dùng toàn bộ, chậm)

    inertia = []
    sil_scores = []
    dbi_scores = []

    for k in K_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=N_INIT_SEARCH)
        labels = km.fit_predict(X_scaled)  # fit 1 lần/K, dùng chung cho cả 3 chỉ số

        inertia.append(km.inertia_)
        dbi_scores.append(davies_bouldin_score(X_scaled, labels))
        sil_scores.append(
            silhouette_score(X_scaled, labels, sample_size=SILHOUETTE_SAMPLE_SIZE, random_state=42)
        )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(K_range, inertia, marker='o')
    axes[0].set_xlabel("K"); axes[0].set_ylabel("WCSS (Inertia)")
    axes[0].set_title("Elbow Method"); axes[0].grid(True)

    axes[1].plot(K_range, sil_scores, marker='o', color='orange')
    axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette Score")
    axes[1].set_title(f"Silhouette Analysis (sample={SILHOUETTE_SAMPLE_SIZE})"); axes[1].grid(True)

    axes[2].plot(K_range, dbi_scores, marker='o', color='green')
    axes[2].set_xlabel("K"); axes[2].set_ylabel("Davies-Bouldin Index")
    axes[2].set_title("Davies-Bouldin Analysis"); axes[2].grid(True)

    plt.tight_layout()
    plt.show()

    best_k = DEFAULT_K  # vẫn dùng K=4 làm quyết định cuối cùng, biểu đồ trên chỉ để tham khảo/đối chiếu
else:
    best_k = DEFAULT_K
    print(f"⏭️  Bỏ qua bước dò K (RUN_K_SELECTION=False) — dùng thẳng K={best_k}.")
    print("   Đổi RUN_K_SELECTION = True ở trên nếu cần chạy lại Elbow/Silhouette/Davies-Bouldin để kiểm chứng.")


In [ ]:
kmeans_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)  # best_k lấy từ cell chọn K phía trên
rfm_model['cluster'] = kmeans_final.fit_predict(X_scaled)

In [ ]:
centroids = rfm_model.groupby('cluster')[['recency_days','purchase_frequency','monetary_total']].mean()


In [ ]:
print("Silhouette:", silhouette_score(X_scaled, rfm_model['cluster']))
print("Davies-Bouldin:", davies_bouldin_score(X_scaled, rfm_model['cluster']))

In [ ]:
# Gán nhãn segment DỰA TRÊN RANK THỰC TẾ của centroid (không hard-code theo cluster index),
# để nhãn luôn khớp với dữ liệu kể cả khi KMeans đổi số hiệu cluster giữa các lần chạy.
#
# SỬA LỖI: bản gốc gán cứng {0:'Cooling down', 1:'Champions/Loyal', 2:'At risk', 3:'New customers'}.
# Nhãn 'New customers' cho cluster 3 SAI với dữ liệu: cluster 3 có recency CAO NHẤT (~241 ngày,
# cao hơn cả 3 cluster còn lại), tức là nhóm LÂU CHƯA QUAY LẠI chứ không phải khách mới.

centroids_rank = centroids.copy()
centroids_rank['frequency_rank'] = centroids_rank['purchase_frequency'].rank(ascending=False)  # 1 = cao nhất
centroids_rank['monetary_rank']  = centroids_rank['monetary_total'].rank(ascending=False)       # 1 = cao nhất
centroids_rank['recency_rank']   = centroids_rank['recency_days'].rank(ascending=True)          # 1 = gần đây nhất

one_timers = centroids_rank[centroids_rank['frequency_rank'] != 1]
# Trong nhóm one-timer, cụm có recency CAO NHẤT (lâu chưa quay lại nhất) -> Dormant/At risk thực sự
dormant_idx = one_timers['recency_days'].idxmax()
remaining = one_timers.drop(index=dormant_idx)

def label_segment(row):
    # Khách mua nhiều lần nhất -> Champions/Loyal
    if row['frequency_rank'] == 1:
        return 'Champions / Loyal'
    one_timers = centroids_rank[centroids_rank['frequency_rank'] != 1]
    if row['monetary_rank'] == one_timers['monetary_rank'].max():
        return 'At risk / Low-value'
    if row['monetary_rank'] == one_timers['monetary_rank'].min():
        return 'High-value (one-time)'
    return 'Mid-value (one-time)'

centroids['segment_label'] = centroids_rank.apply(label_segment, axis=1)
rfm_model['segment_label'] = rfm_model['cluster'].map(centroids['segment_label'])

print(centroids)
print()
print(rfm_model['segment_label'].value_counts())
print()
print("📌 Ghi chú: Recency của cả 4 cluster chỉ dao động trong khoảng hẹp "
      f"({centroids_rank['recency_days'].min():.0f}–{centroids_rank['recency_days'].max():.0f} ngày), "
      "nên KHÔNG dùng recency để đặt tên 'khách mới' — biến R gần như không phân tách được các cụm ở lần "
      "phân cụm này (K đang phân tách chủ yếu theo Frequency và Monetary).")


In [ ]:
# Kiểm tra nhanh: đúng 4 nhãn, không có giá trị null
assert rfm_model['segment_label'].isnull().sum() == 0, "Có customer chưa được gán segment_label!"
print("Số lượng nhãn:", rfm_model['segment_label'].nunique())
print(rfm_model['segment_label'].unique())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
palette_colors = sns.color_palette("Set2", n_colors=rfm_model['segment_label'].nunique())
color_map = dict(zip(rfm_model['segment_label'].unique(), palette_colors))
colors = color_map  # SỬA LỖI: bản gốc dùng biến `colors` nhưng chưa từng định nghĩa -> NameError khi chạy sạch từ đầu

# --- 1. Bar chart: số lượng khách hàng theo segment ---
plt.figure(figsize=(8,5))
order = rfm_model['segment_label'].value_counts().index
sns.countplot(data=rfm_model, x='segment_label', order=order,
              palette=[colors[s] for s in order])
plt.title('Số lượng khách hàng theo Segment')
plt.xlabel('')
plt.ylabel('Số khách hàng')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('segment_count.png', dpi=150)
plt.show()


In [ ]:
# --- 2. Bar chart: trung bình R-F-M theo segment ---
fig, axes = plt.subplots(1, 3, figsize=(16,5))
metrics = ['recency_days', 'purchase_frequency', 'monetary_total']
titles = ['Recency trung bình (ngày)', 'Frequency trung bình', 'Monetary trung bình']

for ax, metric, title in zip(axes, metrics, titles):
    data = rfm_model.groupby('segment_label')[metric].mean().reindex(order)
    ax.bar(data.index, data.values, color=[colors[s] for s in data.index])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('rfm_by_segment.png', dpi=150)
plt.show()


In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['recency_days','frequency_log', 'monetary_log']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_model[features])

In [ ]:
inertia = []
K = range(2,11)

for k in K:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K, inertia, marker='o')
plt.xlabel("Number of Clusters (K)")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

In [ ]:


# --- 3. Scatter plot: Recency vs Monetary, màu theo segment ---
plt.figure(figsize=(8,6))
for seg in order:
    subset = rfm_model[rfm_model['segment_label'] == seg]
    plt.scatter(subset['recency_days'], subset['monetary_total'],
                label=seg, alpha=0.5, s=20, color=colors[seg])
plt.xlabel('Recency (ngày)')
plt.ylabel('Monetary (R$)')
plt.title('Phân bố khách hàng theo Recency & Monetary')
plt.legend()
plt.tight_layout()
plt.savefig('scatter_recency_monetary.png', dpi=150)
plt.show()



In [ ]:
# --- 4. Boxplot: phân phối monetary theo segment (kiểm tra outlier) ---
plt.figure(figsize=(8,5))
sns.boxplot(data=rfm_model, x='segment_label', y='monetary_total',
            order=order, palette=[colors[s] for s in order])
plt.yscale('log')  # vì monetary thường lệch phải mạnh
plt.title('Phân phối Monetary theo Segment (log scale)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('boxplot_monetary.png', dpi=150)
plt.show()


In [ ]:

# --- 5. Pie chart: tỷ lệ % segment ---
plt.figure(figsize=(7,7))
sizes = rfm_model['segment_label'].value_counts().reindex(order)
plt.pie(sizes, labels=sizes.index, autopct='%1.1f%%',
        colors=[colors[s] for s in order], startangle=90)
plt.title('Tỷ lệ khách hàng theo Segment')
plt.tight_layout()
plt.savefig('pie_segment.png', dpi=150)
plt.show()